# Model 3: ConvNeXt_Tiny — Six-Class Facial Skin Classification

**Research:** experiment A (real-only) or B (real+synthetic), always evaluate on **real validation**; frozen test is opt-in at the end. Pretrained backbone + replacement six-class head; head warmup then progressive full fine-tuning. Group-aware splits must already exist.

Kaggle: T4 GPU + Internet for weight download. Set **MANIFEST_CSV** and **IMAGE_ROOT**. Do not run this notebook until the six-class real train/val/test split is frozen. This notebook DOES NOT split the dataset and DOES NOT generate images.

**Dataset contract:** All real images come from the user-uploaded Kaggle v2 collection, after the approved frozen dedup/group split. Never pass the raw unsplit v2 folder into training. The `EDIT_APPROVED_...` input paths must point to your saved Kaggle split outputs. Checkpoint/validation CSVs, final-test class probabilities (only when explicitly unlocked), run logs and SHA-256 artifact inventory are saved under `OUT_DIR`.


**Run evidence:** console output is streamed into `notebook_console.log` from before model loading onward; training history and checkpoint files remain in `OUT_DIR`, with SHA-256 inventory and structured `run_log.jsonl`. All raw logs stay in private Kaggle storage until a privacy-reviewed aggregate export is prepared.


## 1. Install the single additional model library (keep Kaggle PyTorch/CUDA untouched)

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','timm==1.0.20','scikit-learn>=1.4,<2'])

## 2. Editable experiment configuration

In [ ]:
# EDIT ONLY THESE PATHS FIRST (attach Kaggle datasets using Add Input)
MANIFEST_CSV = '/kaggle/input/EDIT_APPROVED_REAL_SPLIT/real_all_manifest.csv'  # needs image_path,label,split,group_id,is_synthetic
IMAGE_ROOT = '/kaggle/input/EDIT_APPROVED_REAL_IMAGES/facial_real_split_v2'       # used for relative image paths
EXPERIMENT = 'real_only'  # 'real_only', 'real_plus_synthetic', 'synthetic_only'
RUN_FINAL_TEST = False   # change ONLY after all hyperparameters and experiment selection are frozen
SEED = 34021
HEAD_EPOCHS = 2
PARTIAL_EPOCHS = 4  # final transformer/CNN stage only
FINETUNE_EPOCHS = 0  # optional full fine-tuning
BATCH_SIZE = 8
NUM_WORKERS = 2
PATIENCE = 4
OUT_DIR = '/kaggle/working/convnext_tiny_outputs'

MODEL_ID='convnext_tiny.fb_in22k_ft_in1k'
IMG_SIZE=224
FINETUNE_LR=5e-05

In [ ]:
# Save all notebook console text, including training metrics and any raised errors.
# Keep raw logs in private Kaggle outputs; publish only aggregate checked CSVs.
import sys,subprocess
from pathlib import Path
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1',
        'https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import start_notebook_logging
PRIVATE_CONSOLE_LOG=start_notebook_logging(OUT_DIR,kind='classifier_training',
    config={'model':'ConvNeXt_Tiny','experiment':EXPERIMENT,'seed':SEED,
            'manifest_sha256':'recorded separately in final artifact inventory',
            'final_real_test_unlocked':RUN_FINAL_TEST},repo_root=DERMSYNREAL_REPO)
print('Persistent Kaggle training console:',PRIVATE_CONSOLE_LOG)


## 3. Inspect the frozen manifest; enforce real group isolation and real-only validation

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, random, json, hashlib
import torch
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, classification_report, roc_auc_score

LABELS = ['acne','fungal_infection','vitiligo','hyperpigmentation','skin_cancer','normal']
LABEL2ID = {s:i for i,s in enumerate(LABELS)}
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device',DEVICE)

m = pd.read_csv(MANIFEST_CSV, dtype=str, keep_default_na=False)
required = {'image_path','label','split','group_id','is_synthetic'}
assert required <= set(m.columns), f'Missing required manifest columns: {required-set(m.columns)}'
for col in required: m[col] = m[col].astype(str).str.strip()
m['label'] = m['label'].str.lower().str.replace(' ', '_', regex=False)
m['split'] = m['split'].str.lower().replace({'validation':'val','valid':'val'})
m['is_synthetic'] = m['is_synthetic'].str.lower().map({'true':True,'1':True,'yes':True,'false':False,'0':False,'no':False})
assert m['is_synthetic'].notna().all(), 'is_synthetic must be true/false or 1/0'
assert m['label'].isin(LABELS).all(), f'Unknown class labels: {set(m.label)-set(LABELS)}'
assert m['split'].isin(['train','val','test']).all(), 'Split must be train, val or test'
assert not m['image_path'].duplicated().any(), 'Duplicate image_path entries'
assert not ((m['is_synthetic']) & (m['split']!='train')).any(), 'Synthetic data must be train only'
real = m[~m['is_synthetic']].copy()
assert not real['group_id'].eq('').any(), 'Every real image requires group_id'
assert {'train','val','test'} <= set(real['split']), 'Real train, val and test required'
group_split_counts = real.groupby('group_id')['split'].nunique()
assert group_split_counts.max() == 1, 'Patient/duplicate group crosses real partitions'
for name in ['train','val']:
 assert set(LABELS) <= set(real.loc[real.split==name,'label']), f'Missing classes in real {name}'
root = Path(IMAGE_ROOT)
def to_path(v):
 p=Path(v)
 return p if p.is_absolute() else root / p
m['resolved_path'] = m['image_path'].map(lambda v:str(to_path(v)))
missing=[p for p in m.resolved_path if not Path(p).is_file()]
assert not missing, f'Missing {len(missing)} files, examples: {missing[:3]}'
if EXPERIMENT == 'real_only':
 tr = m[(m.split=='train') & (~m.is_synthetic)].copy()
elif EXPERIMENT == 'synthetic_only':
 tr=m[(m.split=='train') & (m.is_synthetic)].copy()
 assert len(tr)>0,'E5 requires reviewed synthetic train images'
elif EXPERIMENT == 'real_plus_synthetic':
 tr = m[m.split=='train'].copy()
 assert tr.is_synthetic.any(), 'No synthetic train images in manifest'
else: raise ValueError('EXPERIMENT must be real_only, real_plus_synthetic or synthetic_only')
va = m[(m.split=='val') & (~m.is_synthetic)].copy()
te = m[(m.split=='test') & (~m.is_synthetic)].copy() # evaluated only when RUN_FINAL_TEST=True
TRAIN_STEPS_PER_EPOCH=max(1,(len(real[(real.split=='train')])+BATCH_SIZE-1)//BATCH_SIZE)
print('Matched optimizer updates per epoch:',TRAIN_STEPS_PER_EPOCH)
tr['y']=tr.label.map(LABEL2ID); va['y']=va.label.map(LABEL2ID); te['y']=te.label.map(LABEL2ID)
print('Train',len(tr),'Real val',len(va),'Frozen real test',len(te),'Experiment',EXPERIMENT)

## 4. Initialize pretrained model, resize without cropping, warm up the head, then fine-tune

In [ ]:
import timm, os, copy, time
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import log_loss
out=Path(OUT_DIR);out.mkdir(parents=True,exist_ok=True)
model=timm.create_model(MODEL_ID, pretrained=True, num_classes=len(LABELS)).to(DEVICE)
data_cfg=timm.data.resolve_model_data_config(model)
mean,std=data_cfg['mean'],data_cfg['std']
print('Model:',MODEL_ID,'Input:',IMG_SIZE,'Params:',sum(p.numel() for p in model.parameters()))
# Images are padded, not cropped: preserve peripheral facial lesions.
class PadSquare:
 def __call__(self, im):
  im=ImageOps.exif_transpose(im).convert('RGB'); w,h=im.size
  side=max(w,h); bg=Image.new('RGB',(side,side),(128,128,128)); bg.paste(im,((side-w)//2,(side-h)//2));return bg

def transforms(train):
 aug=[PadSquare(),T.Resize((IMG_SIZE,IMG_SIZE))]
 if train: aug += [T.RandomHorizontalFlip(p=0.5),T.ColorJitter(brightness=.06,contrast=.06,saturation=.04)]
 aug += [T.ToTensor(),T.Normalize(mean,std)]
 return T.Compose(aug)
class FaceDS(Dataset):
 def __init__(self,df,transform):self.df=df.reset_index(drop=True);self.transform=transform
 def __len__(self):return len(self.df)
 def __getitem__(self,i):
  row=self.df.iloc[i]
  with Image.open(row.resolved_path) as f: x=self.transform(f)
  return x,int(row.y)

def loader(df,training):
 return DataLoader(FaceDS(df,transforms(training)),batch_size=BATCH_SIZE,shuffle=training,
                   num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'),drop_last=False)
train_loader=loader(tr,True);val_loader=loader(va,False)
# Only save/load this notebook's own checkpoints. Test labels are not consulted.
head=model.get_classifier()
head_ids={id(p) for p in head.parameters()}
assert head_ids, 'Classifier head not found in timm model'
criterion=nn.CrossEntropyLoss()
use_amp=(DEVICE.type=='cuda')
scaler=torch.amp.GradScaler('cuda',enabled=use_amp)

@torch.no_grad()
def evaluate(dl):
 model.eval(); ys=[];ps=[];ls=[]
 for x,y in dl:
  x=x.to(DEVICE);y=y.to(DEVICE)
  with torch.autocast(device_type=DEVICE.type,enabled=use_amp):
   logits=model(x); loss=criterion(logits,y)
  ls.append((loss.item(),len(y)));ys.extend(y.cpu().tolist());ps.extend(logits.softmax(1).float().cpu().numpy())
 preds=np.asarray(ps).argmax(1); labels=np.asarray(ys)
 return {'macro_f1':float(f1_score(labels,preds,labels=list(range(6)),average='macro',zero_division=0)),
         'balanced_accuracy':float(balanced_accuracy_score(labels,preds)),
         'loss':sum(a*n for a,n in ls)/sum(n for _,n in ls)},labels,np.asarray(ps)

best=-1.;best_file=out/'best_weights.pth';bad=0;history=[]
PARTIAL_PREFIXES=tuple(["stages.3."])
def phase(epochs,stage,lr):
 global best,bad
 if epochs<=0:return
 if stage not in {'head','partial','full'}:raise ValueError(stage)
 for name,p in model.named_parameters():
  is_head=id(p) in head_ids
  is_late=any(name.startswith(pre) for pre in PARTIAL_PREFIXES)
  p.requires_grad=(is_head or (stage=='partial' and is_late) or stage=='full')
 head_params=[p for p in model.parameters() if p.requires_grad and id(p) in head_ids]
 body_params=[p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
 assert head_params and (stage=='head' or body_params),'Could not locate intended final-stage parameters'
 opt=AdamW([{'params':head_params,'lr':lr}]+([{'params':body_params,'lr':lr*.2}] if body_params else []),weight_decay=.02)
 scheduler=CosineAnnealingLR(opt,T_max=max(1,epochs))
 trainable=head_params+body_params
 for epoch in range(epochs):
  if stage=='full':model.train()
  else:
   model.eval();head.train()
   if stage=='partial':
    for mod_name,mod in model.named_modules():
     if any(mod_name==pre.rstrip('.') for pre in PARTIAL_PREFIXES):mod.train()
  epoch_loss=0.;count=0;updates=0
  batch_iterator=iter(train_loader)
  for batch_no in range(TRAIN_STEPS_PER_EPOCH):
   try:
    x,y=next(batch_iterator)
   except StopIteration:
    # E5 may have fewer samples than the real-only reference; repeat
    # shuffled synthetic batches without changing optimizer update count.
    batch_iterator=iter(train_loader)
    x,y=next(batch_iterator)
   x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
   opt.zero_grad(set_to_none=True)
   with torch.autocast(device_type=DEVICE.type,enabled=use_amp):
    logits=model(x);loss=criterion(logits,y)
   scaler.scale(loss).backward();scaler.unscale_(opt)
   nn.utils.clip_grad_norm_(trainable,1.0)
   scaler.step(opt);scaler.update()
   epoch_loss+=loss.item()*len(y);count+=len(y);updates+=1
  assert updates==TRAIN_STEPS_PER_EPOCH,'Insufficient train batches for matched update count'
  scheduler.step()
  v,_,_=evaluate(val_loader)
  history.append({'phase':stage,'epoch':epoch+1,'updates':updates,'train_loss':epoch_loss/max(count,1),**v})
  print(history[-1])
  if v['macro_f1']>best+1e-5:
   best=v['macro_f1'];bad=0;torch.save(model.state_dict(),best_file)
  else:bad+=1
  if bad>=PATIENCE:
   print('Early stopping phase',stage);break

phase(HEAD_EPOCHS,'head',1e-3)
model.load_state_dict(torch.load(best_file,map_location=DEVICE,weights_only=True))
bad=0
phase(PARTIAL_EPOCHS,'partial',FINETUNE_LR)
if FINETUNE_EPOCHS>0:
 model.load_state_dict(torch.load(best_file,map_location=DEVICE,weights_only=True))
 bad=0
 phase(FINETUNE_EPOCHS,'full',FINETUNE_LR*.25)
model.load_state_dict(torch.load(best_file,map_location=DEVICE,weights_only=True))
val_metrics,y_val,p_val=evaluate(val_loader)
print('Best real-validation results:',val_metrics)
pd.DataFrame(history).to_csv(out/'history.csv',index=False)
np.savez_compressed(out/'real_validation_predictions.npz',y=y_val,probabilities=p_val)
(out/'run_config.json').write_text(json.dumps({'model_id':MODEL_ID,'experiment':EXPERIMENT,'labels':LABELS,
  'image_size':IMG_SIZE,'seed':SEED,'finetune_lr':FINETUNE_LR,'partial_epochs':PARTIAL_EPOCHS,'full_epochs':FINETUNE_EPOCHS,'matched_updates_per_epoch':TRAIN_STEPS_PER_EPOCH,'validation':val_metrics,'final_test_run':False},indent=2))
print('Saved',best_file)

## 5. Final real test (locked by default)

In [ ]:
# Keep False during development. A frozen real test should be evaluated only after method selection.
if RUN_FINAL_TEST:
 test_loader=loader(te,False)
 results,y_test,p_test=evaluate(test_loader)
 pred=p_test.argmax(1)
 print('FINAL REAL TEST (do not tune on these results):',results)
 print(classification_report(y_test,pred,labels=list(range(6)),target_names=LABELS,zero_division=0))
 print(confusion_matrix(y_test,pred,labels=list(range(6))))
 np.savez_compressed(out/'FINAL_real_test_predictions.npz',y=y_test,probabilities=p_test)
 cfg=json.loads((out/'run_config.json').read_text());cfg['final_test_run']=True
 (out/'run_config.json').write_text(json.dumps(cfg,indent=2))
else:
 print('Final real test remains locked; set RUN_FINAL_TEST=True only after selection is complete.')

In [ ]:
# Research CSV export: exact existing test-order provenance, no re-inference.
# Only runs after an explicitly enabled final real-test evaluation.
if RUN_FINAL_TEST:
    import hashlib
    artifact = np.load(out / 'FINAL_real_test_predictions.npz')
    test_frame = te.reset_index(drop=True)
    labels_np = np.asarray(artifact['y'], dtype=int)
    scores_np = np.asarray(artifact['probabilities'])
    expected_np = test_frame['y'].to_numpy(dtype=int)
    assert len(test_frame) == len(labels_np) == len(scores_np), 'Test outputs and IDs have different lengths'
    assert np.array_equal(labels_np, expected_np), 'Prediction order does not match frozen real-test manifest'
    assert scores_np.ndim == 2 and scores_np.shape[1] == len(LABELS), 'Unexpected probability shape'
    assert np.all(np.isfinite(scores_np)), 'Nonfinite probabilities detected'
    predictions_table = pd.DataFrame({
        'image_id': test_frame['image_id'].astype(str).to_list() if 'image_id' in test_frame.columns else [hashlib.sha256(str(p).encode('utf-8')).hexdigest() for p in test_frame['image_path']],
        'true_label': [LABELS[int(i)] for i in labels_np],
        'pred_label': [LABELS[int(i)] for i in scores_np.argmax(axis=1)],
        'source': 'real',
        'split': 'test',
        'group_id': test_frame['group_id'].astype(str).to_list(),
        'dataset_source': test_frame['dataset_source'].astype(str).to_list() if 'dataset_source' in test_frame.columns else [''] * len(test_frame),
        'prompt_family_id': [''] * len(test_frame),
    })
    for _index, _class in enumerate(LABELS):
        predictions_table['p_' + _class] = scores_np[:, _index]
    assert predictions_table['image_id'].is_unique
    export_file = out / 'FINAL_real_test_predictions.csv'
    predictions_table.to_csv(export_file, index=False)
    print('Saved paired-evaluation CSV:', export_file, 'rows:', len(predictions_table))


In [ ]:
# Persist reproducible checkpoint/validation history and hashes for later Kaggle evaluation.
from pathlib import Path
import sys,subprocess
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import training_run_report
report=training_run_report(model_output=OUT_DIR,manifest_csv=MANIFEST_CSV,
    model_name=MODEL_ID,
    experiment=EXPERIMENT,seed=SEED,repo_root=DERMSYNREAL_REPO)
print('Artifact inventory:',Path(OUT_DIR)/'artifact_inventory.csv')
print('Run logs:',Path(OUT_DIR)/'run_log.jsonl')
print('Checkpoint provenance:',report['artifacts_sha256'])

### Metric and safety notes
- Macro-F1 is the primary endpoint; class-wise recall and calibration should be evaluated after model selection.
- This is a classification research workflow, **not** a clinical diagnostic device.
- No synthetic image may enter real validation/test. Synthetic prompt-family splitting is handled upstream.
- For a fair experiment B, keep the same architecture/seed and validation/test as A. Run additional seeds and ratio sweeps separately.